In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

inserted_count = 0

try:
    cur = conn.cursor()

    # 📥 엑셀 데이터 로드 및 전처리
    df = pd.read_excel(r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx")
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['price_date'] = pd.to_datetime(df['price_date'])
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)

    for _, row in df.iterrows():
        try:
            # ✅ ticker 존재 확인
            cur.execute("SELECT ticker FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print(f"⛔️ ticker 없음: {row['ticker']}")
                continue

            # ✅ stock_price 확인
            cur.execute("""
                SELECT 1 FROM stock_price
                WHERE ticker = %s AND price_date = %s
            """, (row['ticker'], row['price_date']))
            if cur.fetchone() is None:
                print(f"⛔️ stock_price 없음: {row['ticker']}, {row['price_date']}")
                continue

            # ✅ publisher 처리 (중복 안전하게)
            cur.execute("""
                WITH ins AS (
                    INSERT INTO publisher (name)
                    VALUES (%s)
                    ON CONFLICT (name) DO NOTHING
                    RETURNING publisher_id
                )
                SELECT publisher_id FROM ins
                UNION
                SELECT publisher_id FROM publisher WHERE name = %s;
            """, (row['publisher_name'], row['publisher_name']))
            publisher_id = cur.fetchone()[0]

            # ✅ news 삽입
            cur.execute("""
                INSERT INTO news (
                    ticker, price_date, publisher_id, title, summary, content, url,
                    sentiment, sentiment_score, published_at
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (url) DO NOTHING
                RETURNING news_id
            """, (
                row['ticker'],
                row['price_date'],
                publisher_id,
                row['title'],
                row.get('summary', ''),
                row.get('content', ''),
                row['url'],
                row.get('sentiment', None),
                float(row.get('sentiment_score', 0)),
                row['published_at']
            ))

            result = cur.fetchone()
            if result:
                news_id = result[0]
            else:
                cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                news_id = cur.fetchone()[0]

            # ✅ 키워드 처리 및 연결
            keywords = str(row.get('keywords', '')).split(',')
            for word in keywords:
                word = word.strip()
                if not word:
                    continue

                cur.execute("""
                    WITH ins AS (
                        INSERT INTO keyword (word)
                        VALUES (%s)
                        ON CONFLICT (word) DO NOTHING
                        RETURNING keyword_id
                    )
                    SELECT keyword_id FROM ins
                    UNION
                    SELECT keyword_id FROM keyword WHERE word = %s;
                """, (word, word))
                keyword_id = cur.fetchone()[0]

                cur.execute("""
                    INSERT INTO news_keyword (news_id, keyword_id)
                    VALUES (%s, %s)
                    ON CONFLICT DO NOTHING
                """, (news_id, keyword_id))

            inserted_count += 1

        except Exception as row_error:
            print(f"[❌ row 오류] {row.get('title', '제목없음')} - {row_error}")
            try:
                if conn and conn.closed == 0:
                    conn.rollback()
            except Exception as rollback_error:
                print(f"[⚠️ rollback 실패] {rollback_error}")
            continue

    conn.commit()
    print(f"✅ 총 {inserted_count}건 삽입 완료.")

except Exception as e:
    print(f"[❌ 전체 오류] {e}")

finally:
    try:
        if cur and not cur.closed:
            cur.close()
        if conn and conn.closed == 0:
            conn.close()
    except Exception as close_error:
        print(f"[⚠️ 종료 중 에러] {close_error}")